In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Bibliotecas carregadas com sucesso.")

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

csv_path = data_dir / "B2W-Reviews01.csv"
dataset_url = (
    "https://raw.githubusercontent.com/americanas-tech/"
    "b2w-reviews01/main/B2W-Reviews01.csv"
)

if not csv_path.exists():
    print("Baixando o dataset B2W-Reviews01...")
    try:
        pd.read_csv(dataset_url, sep=";", nrows=5)
        # O arquivo é baixado em blocos para não exigir muita memória adicional.
        import urllib.request
        urllib.request.urlretrieve(dataset_url, csv_path)
        print(f"Arquivo salvo em: {csv_path.resolve()}")
    except Exception as exc:
        raise RuntimeError(
            "Não foi possível baixar o dataset. Baixe B2W-Reviews01.csv "
            "do repositório oficial e salve-o na pasta data."
        ) from exc

# Lemos apenas as colunas necessárias para este exercício.
df = pd.read_csv(
    csv_path,
    usecols=["review_text", "overall_rating"],
    low_memory=False,
)

print(f"Dimensões originais: {df.shape[0]:,} linhas e {df.shape[1]} colunas")
display(df.head())

In [ ]:
import unicodedata

def remover_acentos(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(c)
    )

df = df.dropna(subset=["review_comment"]).copy()
df["review_text"] = df["review_text"].astype(str).str.strip()
df = df[df["review_text"].str.len() > 0]

df["status_entrega"] = np.select(
    [
        df["review_comment"].apply(remover_acentos).str.lower().str.contains(r"nao chegou|entrega nao recebida|entrega nao foi recebida|nao recebi|nao entregou|demorou|atrasou|tardou", case=False, regex=True),
        df["review_comment"].apply(remover_acentos).str.lower().str.contains(r"chegou|nao demorou|rapido|nao |nao atrasou|nao tardou", case=False, regex=True),
    ],
    ["atrasado", "no prazo"],
    default="nao se aplica",
)

# Exclui avaliações neutras neste primeiro experimento.
df = df[df["sentimento"] != "neutro"].copy()

MAX_EXAMPLES = 20000  # experimente 5_000, 20_000, 50_000 ou None
if MAX_EXAMPLES is not None and len(df) > MAX_EXAMPLES:
    # Amostragem estratificada mantém aproximadamente a proporção das classes.
    df = (
        df.groupby("sentimento", group_keys=False)
        .apply(
            lambda grupo: grupo.sample(
                n=max(1, round(MAX_EXAMPLES * len(grupo) / len(df))),
                random_state=RANDOM_STATE,
            )
        )
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )

print(f"Exemplos após preparação: {len(df):,}")
display(df[["review_text", "overall_rating", "sentimento"]].head())

In [ ]:
contagem = df["sentimento"].value_counts()

ax = contagem.plot(kind="bar", figsize=(7, 4))
ax.set_title("Distribuição das classes")
ax.set_xlabel("Sentimento")
ax.set_ylabel("Quantidade de avaliações")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print((contagem / contagem.sum() * 100).round(1).astype(str) + "%")

In [ ]:
for classe in ["positivo", "negativo"]:
    print(f"\n--- Exemplos {classe.upper()} ---")
    exemplos = df.loc[df["sentimento"] == classe, "review_text"].head(3)
    for i, texto in enumerate(exemplos, start=1):
        print(f"{i}. {texto[:500]}")

In [ ]:
X = df["review_text"].astype(str)
y = df["sentimento"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Treinamento: {len(X_train):,} textos")
print(f"Teste:       {len(X_test):,} textos")
print("\nDistribuição no treino:")
print(y_train.value_counts(normalize=True).round(3))
print("\nDistribuição no teste:")
print(y_test.value_counts(normalize=True).round(3))

In [ ]:
vetorizador_demo = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000,
)

# Demonstração pequena para visualizar a representação numérica.
textos_demo = [
    "O produto é muito bom",
    "O produto não é bom",
    "Gostei muito do produto",
]

matriz_demo = vetorizador_demo.fit_transform(textos_demo)
termos_demo = vetorizador_demo.get_feature_names_out()

print("Formato da matriz TF-IDF:", matriz_demo.shape)
display(pd.DataFrame(
    matriz_demo.toarray(),
    columns=termos_demo,
    index=[f"Texto {i+1}" for i in range(len(textos_demo))]
).round(2))

In [ ]:
modelo = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=30000,
            sublinear_tf=True,
        ),
    ),
    (
        "classificador",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
    ),
])

modelo.fit(X_train, y_train)
print("Treinamento concluído.")

In [ ]:
y_pred = modelo.predict(X_test)

print(f"Acurácia: {accuracy_score(y_test, y_pred):.3f}")
print(f"F1 macro: {f1_score(y_test, y_pred, average='macro'):.3f}")
print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=["negativo", "positivo"],
    cmap="Blues",
    values_format="d",
)
plt.title("Matriz de confusão — conjunto de teste")
plt.tight_layout()
plt.show()

In [ ]:
novas_avaliacoes = [
    "Produto excelente, chegou antes do prazo e funciona muito bem.",
    "Péssima qualidade, parou de funcionar no primeiro dia.",
    "O produto é bom, mas a entrega demorou demais.",
    "Não gostei, esperava muito mais.",
    "Atendeu às minhas expectativas.",
    "Muito bonito e fácil de usar.",
]

previsoes = modelo.predict(novas_avaliacoes)

for texto, sentimento in zip(novas_avaliacoes, previsoes):
    print(f"Texto: {texto}")
    print(f"Sentimento previsto: {sentimento}")
    print("-" * 70)

In [ ]:
vet = modelo.named_steps["tfidf"]
clf = modelo.named_steps["classificador"]
termos = vet.get_feature_names_out()
coeficientes = clf.coef_[0]

# A ordem das classes pode variar conforme os rótulos; mostramos também as classes.
print("Classes do classificador:", clf.classes_)

indices_negativos = np.argsort(coeficientes)[:15]
indices_positivos = np.argsort(coeficientes)[-15:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(termos[indices_negativos][::-1], coeficientes[indices_negativos][::-1])
axes[0].set_title("Termos mais associados à primeira classe")
axes[0].set_xlabel("Coeficiente")

axes[1].barh(termos[indices_positivos][::-1], coeficientes[indices_positivos][::-1])
axes[1].set_title("Termos mais associados à segunda classe")
axes[1].set_xlabel("Coeficiente")

plt.tight_layout()
plt.show()

print("Observação: os coeficientes são relativos ao mapeamento de classes do modelo.")